In [ ]:
import json
from pathlib import Path

RESULT_DIR=Path("/rubin/shared/speed-single")

In [ ]:
trimmed_results: dict[str, float|str] = {}
suffix="-results.json"
for res in RESULT_DIR.glob(f"*{suffix}"):
    f_uuid = res.name[:-len(suffix)]
    r_obj = json.loads(res.read_text())
    # Keep the fields we want
    trimmed_results[f_uuid] = {
        "node": r_obj["node"],
        "start": r_obj["start"],
        "end": r_obj["end"],
        "speed": r_obj["real"]  # Only use wallclock
    }

In [ ]:
# Organize trimmed_results by node
results_by_node: dict[str,list[dict[str,float]]] = {}
for v in trimmed_results.values():
    node = v["node"]
    if node not in results_by_node:
        results_by_node[node] = []
    results_by_node[node].append({
        "start": v["start"],
        "end": v["end"],
        "speed": v["speed"]
    })
del trimmed_results    

In [ ]:
# Now stack up overlapping intervals
sorted_results_by_node: dict[str,list[dict[str,float]]] = {}
for node in results_by_node:
    noderesults = sorted(
        results_by_node[node],
        key=lambda x: x["start"]
    )
    sorted_results_by_node[node] = noderesults
del results_by_node

In [ ]:
deltas:dict[str,dict[float,float]] = {}
for node, s_results in sorted_results_by_node.items():
    if node not in deltas:
        deltas[node] = {}
    s_results = sorted_results_by_node[node]
    interval = deltas[node]
    for res in s_results:
        interval[res["start"]] = res["speed"]
        interval[res["end"]] = -res["speed"]
del sorted_results_by_node

In [ ]:
sorted_deltas:dict[str,dict[float,float]] = {}
for node, ds in deltas.items():
    if node not in sorted_deltas:
        sorted_deltas[node] = {}
    interval=sorted_deltas[node]
    keylist = sorted(list(ds.keys()))
    for k in keylist:
        interval[k] = ds[k]
del deltas

In [ ]:
speeds:dict[str,dict[float,float]] = {}
current=0
for node, sds in sorted_deltas.items():
    if node not in speeds:
        speeds[node] = {}
    interval=speeds[node]
    for tstamp, change in sds.items():
        current += change
        interval[tstamp] = current
del sorted_deltas

In [ ]:
speed_points:dict[str,list[tuple[float]]] = {}
for node, spds in speeds.items():
    if node not in speed_points:
        speed_points[node] = []
    interval = speed_points[node]
    for tstamp, speed in spds.items():
        interval.append( (tstamp, speed) )
del speeds

In [ ]:
plot_points:dict[str,dict[str,list[float]]] = {}
for node, sps in speed_points.items():
    if node not in plot_points:
        plot_points[node] = {}
    interval = plot_points[node]
    times = [ x[0] for x in sps]
    speeds = [ x[1] for x in sps]
    interval["times"] = times
    interval["speeds"] = speeds


In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import datetime

fig,axs = plt.subplots(nrows=len(list(plot_points.keys())), ncols=1, sharex=True, figsize=(9,4))
nodenum=0
plt.xticks(rotation=25)
plt.ylabel("MBPs")
for node, val in plot_points.items():
    try:
        _ = len(axs)
    except TypeError:
        axs=[axs]
    dts = [ datetime.datetime.fromtimestamp(x, tz=datetime.UTC) for x in val["times"]]
    axs[nodenum].plot(dts, val["speeds"])
    max_speed=max(val["speeds"])
    idx=val["speeds"].index(max_speed)
    date=datetime.datetime.fromtimestamp(val["times"][idx],tz=datetime.UTC)
    max_speed_s='{0:.2f}'.format(max(val["speeds"]))
    print(f"Node {node}: max_speed={max_speed_s} MBps at {date}")
    nodenum += 1